In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class DoubleConv(nn.Module):
    """(Conv3x3 -> BatchNorm -> ReLU) x 2 """
     
     
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=out_channels, out_channels=out_channels,kernel_size = 3, padding=1),
            nn.BatchNorm2d(num_features = out_channels),
            nn.ReLU(inplace=True),
        )
        
    def forward(self, x):
        return self.net(x)
    
    

class EncoderBlock(nn.Module):
    """Downsampling: it increases chanbels and decrease height and width"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
    def forward(self, input):
        skip = self.conv(input)
        pool = self.pool(skip)
        return skip, pool
        
    
        
class DecoderBlock(nn.Module):
    """Upsampling, increasing height and weight and decreasing channel 
       we perform transpose convolution(inverse of Maxpool) followed by double convolution
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2, padding=0)
        self.conv = DoubleConv(out_channels*2, out_channels)
        
    def forward(self, input, skip):
        x = self.up(input)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)
        

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.E1 = EncoderBlock(3, 64)
        self.E2 = EncoderBlock(64, 128)
        self.E3 = EncoderBlock(128, 256)
        self.E4 = EncoderBlock(256, 512)
        
        self.bottleneck = DoubleConv(512, 1024)
        self.D1 = DecoderBlock(1024, 512)
        self.D2 = DecoderBlock(512, 256)
        self.D3 = DecoderBlock(256, 128)
        self.D4 = DecoderBlock(128,64)
        
        self.final = nn.Conv2d(in_channels=64, out_channels=1, kernel_size=1, stride=1, padding=0, bias=True)
        
        def forward(self, images):
            skip1, pool1 = self.E1(images)
            skip2, pool2 = self.E2(pool1)
            skip3, pool3 = self.E3(pool2)
            skip4, pool4 = self.E4(pool3)
            
            bottleneck = self.bottleneck(pool4) 
            up1=self.D1(bottleneck, skip4)
            up2 = self.D2(up1, skip3 )
            up3 = self.D3(up2, skip2)
            up4 = self.D4(up3, skip1)
            
            
            return self.final(up4)
        
